In [1]:
# Randomized Survey Creator for Blind Audio Evaluation
import shutil
import json
import random
import time
from pathlib import Path

In [2]:
# Create Randomized Anonymous Survey with Descriptive Names
source_dir = "/home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/outputs/survey_offlines"
survey_dir = "/home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/anonymous_survey"
mapping_file = "/home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/anonymous_survey/survey_mapping.json"

source_path = Path(source_dir)
survey_path = Path(survey_dir)

# Clean up existing survey and create fresh
if survey_path.exists():
    shutil.rmtree(survey_path)
survey_path.mkdir(exist_ok=True)

print("🎲 CREATING RANDOMIZED SURVEY with Descriptive Names")
print("=" * 50)

# Get scripts
scripts = [d for d in source_path.iterdir() if d.is_dir()]

# Get test sets
first_script = scripts[0]
test_sets = [d for d in first_script.iterdir() if d.is_dir()]
print(f"📁 Test sets: {[ts.name for ts in test_sets]}")

# Master mapping to track all assignments
master_mapping = {
    "test_mappings": {},
    "filename_mappings": {}  # Maps survey filename -> original filename
}
total_files = 0

# Process each test set
test_counters = {}  # Track test numbers per test set

for test_set in test_sets:
    test_set_name = test_set.name
    
    # Get short name for test set (first few chars)
    if "aria" in test_set_name.lower():
        short_name = "aria"
    elif "pop909" in test_set_name.lower():
        short_name = "pop909"
    elif "test" in test_set_name.lower():
        short_name = "test"
    else:
        short_name = test_set_name[:5]  # First 5 chars as fallback
    
    test_counters[short_name] = 0
    
    test_files = [f for f in test_set.iterdir() if f.suffix in ['.mp3']]
    print(f"\n⚡ {test_set_name}: {len(test_files)} tests -> {short_name}_X")
    
    # Create survey files - RANDOMIZE EACH TEST INDIVIDUALLY
    for test_file in test_files:
        test_counters[short_name] += 1
        test_num = test_counters[short_name]
        
        original_name = test_file.name
        file_ext = test_file.suffix
        
        # RANDOMIZE A/B/C for THIS specific test
        letters = ['A', 'B', 'C']
        random.shuffle(letters)
        
        # Create mapping for this specific test
        test_mapping = {}
        filename_mapping = {}
        
        for i, script in enumerate(scripts):
            letter = letters[i]
            test_mapping[script.name] = letter
            
            # Create descriptive filename: testset_numberLetter.extension
            survey_filename = f"{short_name}_{test_num}{letter}{file_ext}"
            
            # Map survey filename back to original
            filename_mapping[survey_filename] = {
                "original_filename": original_name,
                "script": script.name,
                "test_set": test_set_name,
                "original_path": f"survey_offlines/{script.name}/{test_set_name}/{original_name}"
            }
        
        # Store mappings
        test_key = f"{short_name}_{test_num}"
        master_mapping["test_mappings"][test_key] = test_mapping
        master_mapping["filename_mappings"].update(filename_mapping)
        
        # Copy files with descriptive names
        for script in scripts:
            letter = test_mapping[script.name]
            script_test_set = script / test_set_name
            source_file = script_test_set / original_name
            
            if source_file.exists():
                survey_filename = f"{short_name}_{test_num}{letter}{file_ext}"
                dest_file = survey_path / survey_filename
                shutil.copy2(source_file, dest_file)
                total_files += 1
            else:
                print(f"    ⚠️  Missing: {source_file}")

# Save master mapping
master_mapping["total_files"] = total_files
master_mapping["total_test_sets"] = len(test_sets)
master_mapping["total_tests"] = len(master_mapping["test_mappings"])

with open(mapping_file, 'w') as f:
    json.dump(master_mapping, f, indent=2)

print(f"\n✅ Survey created: {survey_dir}")
print(f"🎵 {total_files} files in {len(master_mapping['test_mappings'])} tests")
print(f"📋 Mapping: {mapping_file}")

# Show sample filenames and mappings
print(f"\n📝 SAMPLE FILENAMES:")
sample_files = list(master_mapping["filename_mappings"].items())[:6]
for survey_name, info in sample_files:
    print(f"   {survey_name} -> {info['original_filename']} ({info['script']})")

print(f"\n🎲 SAMPLE TEST RANDOMIZATIONS:")
sample_tests = list(master_mapping["test_mappings"].items())[:3]
for test_name, mapping in sample_tests:
    print(f"   {test_name}: {mapping}")

🎲 CREATING RANDOMIZED SURVEY with Descriptive Names
📁 Test sets: ['aria_unique_skyline_top2_subset_5', 'test_set', 'pop909_dataset_subset_5']

⚡ aria_unique_skyline_top2_subset_5: 5 tests -> aria_X

⚡ test_set: 5 tests -> test_X

⚡ pop909_dataset_subset_5: 5 tests -> pop909_X

✅ Survey created: /home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/anonymous_survey
🎵 45 files in 15 tests
📋 Mapping: /home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/anonymous_survey/survey_mapping.json

📝 SAMPLE FILENAMES:
   aria_1B.mp3 -> data_bw_097718_0output.mp3 (fake_offline_script_lantency2)
   aria_1C.mp3 -> data_bw_097718_0output.mp3 (fake_offline_script)
   aria_1A.mp3 -> data_bw_097718_0output.mp3 (real_offline_script)
   aria_2A.mp3 -> data_at_039726_0output.mp3 (fake_offline_script_lantency2)
   aria_2C.mp3 -> data_at_039726_0output.mp3 (fake_offline_script)
   aria_2B.mp3 -> data_at_039726_0output.mp3 (real_offline_script)

🎲 SAMPLE TEST RANDOMIZATIONS:
   aria_1: {'fake_offline_sc

In [3]:
# View Survey Structure and Detailed Mappings
survey_path = Path("/home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/anonymous_survey")

if survey_path.exists() and Path("/home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/anonymous_survey/survey_mapping.json").exists():
    # Load mapping
    with open("/home/ubuntu/ugrip/andrew/StreamMUSE/inference_benchmark/anonymous_survey/survey_mapping.json", 'r') as f:
        mapping = json.load(f)
    
    print("📋 SURVEY STRUCTURE:")
    print(f"   Total tests: {mapping['total_tests']}")
    print(f"   Total files: {mapping['total_files']}")
    
    # Show actual files created
    survey_files = list(survey_path.glob("*"))
    survey_files.sort()
    
    print(f"\n📁 FILES CREATED ({len(survey_files)} files):")
    
    # Group by test set
    by_testset = {}
    for file in survey_files:
        if file.is_file():
            # Extract test set from filename (aria_1A -> aria)
            parts = file.stem.split('_')
            if len(parts) >= 2:
                testset = parts[0]
                if testset not in by_testset:
                    by_testset[testset] = []
                by_testset[testset].append(file.name)
    
    # Display grouped files
    for testset, files in by_testset.items():
        print(f"   {testset.upper()}: {len(files)} files")
        for file in sorted(files)[:6]:  # Show first 6
            if file in mapping.get("filename_mappings", {}):
                original = mapping["filename_mappings"][file]["original_filename"]
                script = mapping["filename_mappings"][file]["script"]
                print(f"     {file} <- {original} ({script})")
        if len(files) > 6:
            print(f"     ... and {len(files) - 6} more")
        print()
    
    print("🎲 RANDOMIZATION EXAMPLES:")
    sample_tests = list(mapping.get("test_mappings", {}).items())[:3]
    for test_name, test_mapping in sample_tests:
        print(f"   {test_name}: {test_mapping}")
        
else:
    print("❌ Survey not found. Run cell 2 first.")

📋 SURVEY STRUCTURE:
   Total tests: 15
   Total files: 45

📁 FILES CREATED (46 files):
   ARIA: 15 files
     aria_1A.mp3 <- data_bw_097718_0output.mp3 (real_offline_script)
     aria_1B.mp3 <- data_bw_097718_0output.mp3 (fake_offline_script_lantency2)
     aria_1C.mp3 <- data_bw_097718_0output.mp3 (fake_offline_script)
     aria_2A.mp3 <- data_at_039726_0output.mp3 (fake_offline_script_lantency2)
     aria_2B.mp3 <- data_at_039726_0output.mp3 (real_offline_script)
     aria_2C.mp3 <- data_at_039726_0output.mp3 (fake_offline_script)
     ... and 9 more

   POP909: 15 files
     pop909_1A.mp3 <- 440output.mp3 (fake_offline_script)
     pop909_1B.mp3 <- 440output.mp3 (fake_offline_script_lantency2)
     pop909_1C.mp3 <- 440output.mp3 (real_offline_script)
     pop909_2A.mp3 <- 343output.mp3 (fake_offline_script)
     pop909_2B.mp3 <- 343output.mp3 (fake_offline_script_lantency2)
     pop909_2C.mp3 <- 343output.mp3 (real_offline_script)
     ... and 9 more

   SURVEY: 1 files

   TEST: 15